# 04 Train Baseline LSTM Model Notebook

This notebook trains the first baseline ASL sign recognition model using the cleaned WLASL100 MediaPipe keypoint dataset. The model uses a BiLSTM architecture to learn movement patterns from landmark sequences and classify each sample into one of 100 ASL glosses.

## Import Libraries

In [11]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from tqdm.auto import tqdm
from sklearn.metrics import f1_score
import time
import torch.nn.functional as F

e:\Be_My_Ear\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Set seed and paths

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = Path("E:/Be_My_Ear")

BASE_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / "WLASL100"
CLEAN_INDEX_FILE = BASE_DIR / "wlasl100_clean_keypoint_index.csv"
LABEL_MAP_FILE = PROJECT_ROOT / "data" / "label_maps" / "asl_wlasl100_labels.json"

MODEL_DIR = PROJECT_ROOT / "models" / "ASL"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "baseline_bilstm_wlasl100.pt"

print("Clean index exists:", CLEAN_INDEX_FILE.exists())
print("Label map exists:", LABEL_MAP_FILE.exists())
print("Model will save to:", MODEL_PATH)

Clean index exists: True
Label map exists: True
Model will save to: E:\Be_My_Ear\models\ASL\baseline_bilstm_wlasl100.pt


## Load dataset index

In [3]:
df = pd.read_csv(CLEAN_INDEX_FILE)

print("Total samples:", len(df))
print("Total classes:", df["label_id"].nunique())
print("Input file example:", df.iloc[0]["keypoint_path"])

df.head()

Total samples: 1013
Total classes: 100
Input file example: E:\Be_My_Ear\data\processed\ASL\WLASL100\keypoints\69422.npy


,video_id,gloss,label_id,original_class_id,video_path,keypoint_path,zero_ratio
0,69422,orange,67,27,E:\Be_My_Ear\data\raw\ASL\videos\69422.mp4,E:\Be_My_Ear\data\processed\ASL\WLASL100\keypo...,0.321512
1,10898,city,20,82,E:\Be_My_Ear\data\raw\ASL\videos\10898.mp4,E:\Be_My_Ear\data\processed\ASL\WLASL100\keypo...,0.126163
2,10893,city,20,82,E:\Be_My_Ear\data\raw\ASL\videos\10893.mp4,E:\Be_My_Ear\data\processed\ASL\WLASL100\keypo...,0.264535
3,10892,city,20,82,E:\Be_My_Ear\data\raw\ASL\videos\10892.mp4,E:\Be_My_Ear\data\processed\ASL\WLASL100\keypo...,0.170930
4,10895,city,20,82,E:\Be_My_Ear\data\raw\ASL\videos\10895.mp4,E:\Be_My_Ear\data\processed\ASL\WLASL100\keypo...,0.464341


## Create train, validation, test split

Because some classes only have 5 samples, I use a manual class-balanced split.

In [4]:
train_records = []
val_records = []
test_records = []

for label_id, group in df.groupby("label_id"):
    group = group.sample(frac=1, random_state=SEED).reset_index(drop=True)

    n = len(group)

    n_test = max(1, int(round(n * 0.15)))
    n_val = max(1, int(round(n * 0.15)))

    test_part = group.iloc[:n_test]
    val_part = group.iloc[n_test:n_test + n_val]
    train_part = group.iloc[n_test + n_val:]

    train_records.append(train_part)
    val_records.append(val_part)
    test_records.append(test_part)

train_df = pd.concat(train_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(val_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = pd.concat(test_records).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

print("Train classes:", train_df["label_id"].nunique())
print("Validation classes:", val_df["label_id"].nunique())
print("Test classes:", test_df["label_id"].nunique())

Train samples: 697
Validation samples: 158
Test samples: 158
Train classes: 100
Validation classes: 100
Test classes: 100


## PyTorch Dataset

In [13]:
class SignKeypointDataset(Dataset):
    def __init__(self, dataframe, normalize=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.normalize = normalize

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        keypoints = np.load(row["keypoint_path"]).astype(np.float32)

        if self.normalize:
            mean = keypoints.mean(axis=0, keepdims=True)
            std = keypoints.std(axis=0, keepdims=True)
            keypoints = (keypoints - mean) / (std + 1e-6)

        label = int(row["label_id"])

        keypoints = torch.tensor(keypoints, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.long)

        return keypoints, label

## Create Dataloaders

In [ ]:
BATCH_SIZE = 32

train_dataset = SignKeypointDataset(train_df, normalize=True)
val_dataset = SignKeypointDataset(val_df, normalize=True)
test_dataset = SignKeypointDataset(test_df, normalize=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

x_batch, y_batch = next(iter(train_loader))

print("Input batch shape:", x_batch.shape)
print("Label batch shape:", y_batch.shape)
print("Example labels:", y_batch[:10])

Input batch shape: torch.Size([32, 60, 258])
Label batch shape: torch.Size([32])


## Implement model

### Build BiLSTM model

In [7]:
class SignBiLSTM(nn.Module):
    def __init__(self, input_size=258, hidden_size=256, num_classes=100, num_layers=2, dropout=0.3):
        super(SignBiLSTM, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        logits = self.classifier(last_output)
        return logits

### Set device, model, loss, optimiser

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

NUM_CLASSES = df["label_id"].nunique()

model = SignBiLSTM(
    input_size=258,
    hidden_size=256,
    num_classes=NUM_CLASSES,
    num_layers=2,
    dropout=0.3
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model

Using device: cpu


SignBiLSTM(
  (lstm): LSTM(258, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (classifier): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=100, bias=True)
  )
)

### Training and evaluation functions

In [9]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0
    all_preds = []
    all_labels = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return avg_loss, acc, f1


def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            outputs = model(x)
            loss = criterion(outputs, y)

            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return avg_loss, acc, f1

### Train Model

Initial with 20 epoch then increase

In [10]:
EPOCHS = 20

history = {
    "train_loss": [],
    "train_acc": [],
    "train_f1": [],
    "val_loss": [],
    "val_acc": [],
    "val_f1": []
}

best_val_f1 = 0

for epoch in range(EPOCHS):
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_loss, val_acc, val_f1 = evaluate(
        model, val_loader, criterion, device
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), MODEL_PATH)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f} "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}"
    )

print("Best validation F1:", best_val_f1)
print("Model saved to:", MODEL_PATH)

e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\utils\multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 5

Epoch [1/20] Train Loss: 4.6158 | Train Acc: 0.0043 | Train F1: 0.0017 Val Loss: 4.6036 | Val Acc: 0.0063 | Val F1: 0.0002


e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\utils\multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 5

Epoch [2/20] Train Loss: 4.6098 | Train Acc: 0.0043 | Train F1: 0.0005 Val Loss: 4.6011 | Val Acc: 0.0190 | Val F1: 0.0045


e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\utils\multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 5

...

e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\utils\multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 5

Epoch [18/20] Train Loss: 3.9531 | Train Acc: 0.0301 | Train F1: 0.0128 Val Loss: 4.3206 | Val Acc: 0.0316 | Val F1: 0.0059


e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\utils\multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 5

Epoch [19/20] Train Loss: 3.9409 | Train Acc: 0.0430 | Train F1: 0.0210 Val Loss: 4.3546 | Val Acc: 0.0253 | Val F1: 0.0035
Epoch [20/20] Train Loss: 3.9013 | Train Acc: 0.0459 | Train F1: 0.0302 Val Loss: 4.4143 | Val Acc: 0.0380 | Val F1: 0.0082
Best validation F1: 0.011522624434389141
Model saved to: E:\Be_My_Ear\models\ASL\baseline_bilstm_wlasl100.pt


e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\utils\multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
e:\Be_My_Ear\.venv\lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 5